# Local Iceberg Demo

This notebook lists and queries the five seeded Iceberg tables in the `demo` namespace.

Prerequisites:
1. `docker compose up -d`
2. `python scripts/create_test_tables.py`

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "scripts"))

from spark_session import CATALOG_NAME, create_spark

spark = create_spark("iceberg-local-demo")

In [ ]:
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.demo").show(truncate=False)

In [ ]:
spark.sql(
    f"""
    SELECT
        c.name AS customer_name,
        o.order_id,
        o.status,
        p.name AS product_name,
        oi.quantity,
        s.carrier
    FROM {CATALOG_NAME}.demo.orders o
    JOIN {CATALOG_NAME}.demo.customers c ON o.customer_id = c.customer_id
    JOIN {CATALOG_NAME}.demo.order_items oi ON o.order_id = oi.order_id
    JOIN {CATALOG_NAME}.demo.products p ON oi.product_id = p.product_id
    LEFT JOIN {CATALOG_NAME}.demo.shipments s ON o.order_id = s.order_id
    ORDER BY o.order_id
    """
).show(truncate=False)

In [ ]:
spark.stop()